# Class 4 — Long-Term Memory & LangGraph Agents

**Week 5: Introduction to AI Agents**

### Learning objectives
By the end of this notebook you will be able to:
- Distinguish **short-term** graph checkpoints from **long-term** JSON notes that survive new threads.
- Build a linear LangGraph with explicit **state**, **nodes**, and **edges**.
- Persist durable facts to a local JSON file and recall them on later invokes.
- Wire a simple **multi-agent** pipeline (Researcher → Writer) over a mock game catalog.
- Know when Week 6 vector/RAG memory is the next step — we do **not** use embeddings here.

## Setup

In [ ]:
!pip install -q langgraph langchain-groq

In [ ]:
import json
import os
import re
import uuid

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
try:
    from google.colab import userdata
    GROQ_API_KEY = GROQ_API_KEY or userdata.get("GROQ_API_KEY")
except Exception:
    pass

if not GROQ_API_KEY:
    print("No GROQ_API_KEY found. Live LLM nodes will skip.")
else:
    print("Found GROQ_API_KEY. LangGraph demos are ready.")

from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from typing import TypedDict


def make_llm():
    if not GROQ_API_KEY:
        return None
    return ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=GROQ_API_KEY)

---

## Part A — Linear long-term memory graph

**Flow:** `remember → recall → answer`

- **Short-term:** `InMemorySaver` keeps graph state per `thread_id` for this process.
- **Long-term:** `agent_notes.json` survives new threads and notebook restarts.

We write durable facts in `remember`, load them in `recall`, and ground the Groq reply in `answer`.

### A.1 JSON note helpers

In [ ]:
NOTES_PATH = "agent_notes.json"


def load_notes(path: str = NOTES_PATH) -> dict:
    # Load long-term notes from JSON; return {} if missing.
    if not os.path.exists(path):
        return {}
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def save_note(key: str, value: str, path: str = NOTES_PATH) -> dict:
    # Merge one key into the JSON store and return the full notes dict.
    notes = load_notes(path)
    notes[key] = value
    with open(path, "w", encoding="utf-8") as f:
        json.dump(notes, f, indent=2)
    return notes

### A.2 Graph state and nodes

In [ ]:
class MemoryState(TypedDict):
    user_message: str
    notes: dict
    reply: str


def _extract_preference(message: str) -> tuple[str, str] | None:
    # Heuristic: pull durable facts like 'My favorite game is Celeste'.
    patterns = [
        r"my favorite game is ([^.!?
]+)",
        r"remember that my favorite game is ([^.!?
]+)",
        r"i prefer ([^.!?
]+) as my favorite game",
        r"my favorite color is ([^.!?
]+)",
        r"remember that (.+)",
    ]
    lower = message.lower().strip()
    for pattern in patterns:
        match = re.search(pattern, lower)
        if match:
            value = match.group(1).strip().strip('"'')
            if "favorite game" in pattern or "prefer" in pattern:
                return ("favorite_game", value.title())
            if "favorite color" in pattern:
                return ("favorite_color", value)
            return ("note", value)
    return None


def remember(state: MemoryState) -> dict:
    # If the user states a durable fact, persist it to JSON.
    extracted = _extract_preference(state["user_message"])
    if extracted:
        key, value = extracted
        save_note(key, value)
        print(f"Saved long-term note: {key} = {value}")
    return {}


def recall(state: MemoryState) -> dict:
    # Load JSON notes into graph state.
    return {"notes": load_notes()}


def answer(state: MemoryState) -> dict:
    # Groq reply grounded in recalled notes.
    llm = make_llm()
    notes = state.get("notes") or {}
    if llm is None:
        preview = json.dumps(notes) if notes else "(empty)"
        return {"reply": f"Skipping LLM — no GROQ_API_KEY. Recalled notes: {preview}"}

    notes_block = json.dumps(notes, indent=2) if notes else "(no saved notes yet)"
    prompt = (
        "You are a helpful assistant with access to long-term notes about the user.
"
        f"Known notes:
{notes_block}

"
        f"User message: {state['user_message']}

"
        "Answer concisely. Use the notes when relevant; do not invent stored facts."
    )
    response = llm.invoke(prompt)
    return {"reply": response.content}


def build_memory_graph():
    graph = StateGraph(MemoryState)
    graph.add_node("remember", remember)
    graph.add_node("recall", recall)
    graph.add_node("answer", answer)
    graph.add_edge(START, "remember")
    graph.add_edge("remember", "recall")
    graph.add_edge("recall", "answer")
    graph.add_edge("answer", END)
    return graph.compile(checkpointer=InMemorySaver())

### A.3 Demo — save a preference, recall on a new thread

In [ ]:
memory_graph = build_memory_graph()

turn1_msg = "My favorite game is Celeste."
turn2_msg = "What is my favorite game?"

if memory_graph is None:
    print("Skipping Part A demo — graph not built.")
else:
    # Turn 1: save preference (any thread_id)
    config1 = {"configurable": {"thread_id": f"memory-a-{uuid.uuid4()}"}}
    result1 = memory_graph.invoke(
        {"user_message": turn1_msg, "notes": {}, "reply": ""},
        config=config1,
    )
    print("=== Turn 1 (save preference) ===")
    print(result1["reply"])

    # Turn 2: new thread_id — JSON file still has the fact
    config2 = {"configurable": {"thread_id": f"memory-b-{uuid.uuid4()}"}}
    result2 = memory_graph.invoke(
        {"user_message": turn2_msg, "notes": {}, "reply": ""},
        config=config2,
    )
    print("
=== Turn 2 (new thread, recall from JSON) ===")
    print(result2["reply"])
    print("
On-disk notes:", load_notes())

---

## Part B — Game reporter multi-agent

**Flow:** `researcher → writer`

The Researcher looks up a game in a **mock catalog** (no web scrape, no vector DB).
The Writer drafts a short blog post using **only** the research string — no invented facts.

### B.1 Mock game catalog

In [ ]:
GAME_CATALOG = {
    "celeste": {
        "name": "Celeste",
        "genre": "Platformer",
        "year": 2018,
        "facts": [
            "Players climb Celeste Mountain as Madeline, facing platforming challenges and inner doubts.",
            "Tight controls and forgiving assist modes make it approachable for many skill levels.",
            "The soundtrack by Lena Raine is widely praised.",
            "Strawberries and B-side chapters add optional challenge.",
        ],
    },
    "hades": {
        "name": "Hades",
        "genre": "Roguelike action",
        "year": 2020,
        "facts": [
            "You play Zagreus, son of Hades, trying to escape the underworld.",
            "Each run unlocks new weapons, boons, and story dialogue.",
            "Supergiant Games shipped a polished narrative roguelike loop.",
            "Voice acting and art direction are standout features.",
        ],
    },
    "stardew valley": {
        "name": "Stardew Valley",
        "genre": "Farming sim",
        "year": 2016,
        "facts": [
            "You inherit a run-down farm and rebuild community ties in Pelican Town.",
            "Seasons, crops, mining, and festivals drive the calendar loop.",
            "Created largely solo by Eric Barone (ConcernedApe).",
            "Co-op farming is supported on many platforms.",
        ],
    },
    "among us": {
        "name": "Among Us",
        "genre": "Social deduction",
        "year": 2018,
        "facts": [
            "Crewmates complete tasks while impostors sabotage and eliminate players.",
            "Emergency meetings and voting drive social deduction moments.",
            "InnerSloth developed the breakout multiplayer hit.",
            "Cross-platform play helped its 2020 popularity surge.",
        ],
    },
    "minecraft": {
        "name": "Minecraft",
        "genre": "Sandbox",
        "year": 2011,
        "facts": [
            "Procedural worlds built from blocks support survival and creative modes.",
            "Crafting, redstone, and modding ecosystems extend replayability.",
            "Originally created by Markus Persson (Notch); now owned by Microsoft.",
            "Multiplayer servers range from minigames to massive builds.",
        ],
    },
    "zelda breath of the wild": {
        "name": "The Legend of Zelda: Breath of the Wild",
        "genre": "Action-adventure",
        "year": 2017,
        "facts": [
            "Open-world Hyrule encourages exploration, shrines, and emergent combat.",
            "Physics and chemistry systems enable creative problem solving.",
            "Launched with Nintendo Switch; won broad critical acclaim.",
            "Sequel Tears of the Kingdom expands building and vertical exploration.",
        ],
    },
    "valorant": {
        "name": "Valorant",
        "genre": "Tactical FPS",
        "year": 2020,
        "facts": [
            "5v5 rounds combine precise gunplay with agent abilities.",
            "Riot Games designed it for competitive esports.",
            "Economy and ability cooldowns shape each round.",
            "Regular agent and map updates keep the meta shifting.",
        ],
    },
    "balatro": {
        "name": "Balatro",
        "genre": "Roguelike deckbuilder",
        "year": 2024,
        "facts": [
            "Poker hands score runs in a roguelike loop with jokers and modifiers.",
            "LocalThunk developed the breakout indie hit.",
            "Synergistic joker builds create escalating combo potential.",
            "Short runs and high scores encourage 'one more attempt'.",
        ],
    },
}


def lookup_game(name: str) -> dict | None:
    # Normalize and fuzzy-match a catalog entry by key or display name.
    query = name.strip().lower()
    if not query:
        return None
    for key, game in GAME_CATALOG.items():
        title = game["name"].lower()
        if query == key or query in title or title in query or key in query:
            return game
    return None


def format_research(game: dict) -> str:
    bullets = "
".join(f"- {fact}" for fact in game["facts"])
    return (
        f"Game: {game['name']}
"
        f"Genre: {game['genre']}
"
        f"Year: {game['year']}
"
        f"Facts:
{bullets}"
    )

### B.2 Reporter graph — researcher → writer

In [ ]:
class ReporterState(TypedDict):
    user_message: str
    research: str
    blog_post: str


def _extract_game_query(message: str) -> str:
    # Pull a game title from prompts like 'Write a post about Celeste'.
    patterns = [
        r"write a post about ([^.!?
]+)",
        r"blog post about ([^.!?
]+)",
        r"report on ([^.!?
]+)",
        r"about ([^.!?
]+)",
    ]
    lower = message.lower().strip()
    for pattern in patterns:
        match = re.search(pattern, lower)
        if match:
            return match.group(1).strip().strip('"'')
    return message.strip()


def researcher(state: ReporterState) -> dict:
    query = _extract_game_query(state["user_message"])
    game = lookup_game(query)
    if game is None:
        return {
            "research": (
                f"RESEARCH MISS: No catalog entry for '{query}'. "
                "Do not invent facts — ask the user to clarify the game title."
            )
        }
    return {"research": format_research(game)}


def writer(state: ReporterState) -> dict:
    research = state.get("research", "")
    if research.startswith("RESEARCH MISS"):
        return {"blog_post": research}

    llm = make_llm()
    if llm is None:
        return {"blog_post": "Skipping writer — no GROQ_API_KEY.

Research notes:
" + research}

    prompt = (
        "You are a game blogger. Draft a short blog post (3–5 sentences) using ONLY the research notes below.
"
        "Do NOT invent genres, years, or facts that are not in the notes.

"
        f"Research notes:
{research}

"
        f"Original request: {state['user_message']}"
    )
    response = llm.invoke(prompt)
    return {"blog_post": response.content}


def build_reporter_graph():
    graph = StateGraph(ReporterState)
    graph.add_node("researcher", researcher)
    graph.add_node("writer", writer)
    graph.add_edge(START, "researcher")
    graph.add_edge("researcher", "writer")
    graph.add_edge("writer", END)
    return graph.compile(checkpointer=InMemorySaver())

### B.3 Demo — Celeste hit and unknown-game miss

In [ ]:
reporter_graph = build_reporter_graph()

celeste_request = "Write a post about Celeste"
unknown_request = "Write a post about Nonexistent Game XYZ"

if reporter_graph is None:
    print("Skipping Part B demo — graph not built.")
else:
    cfg = {"configurable": {"thread_id": f"reporter-{uuid.uuid4()}"}}

    print("=== Celeste (catalog hit) ===")
    celeste_result = reporter_graph.invoke(
        {"user_message": celeste_request, "research": "", "blog_post": ""},
        config=cfg,
    )
    print("Research:
", celeste_result["research"])
    print("
Blog draft:
", celeste_result["blog_post"])

    print("
=== Unknown game (miss path) ===")
    miss_result = reporter_graph.invoke(
        {"user_message": unknown_request, "research": "", "blog_post": ""},
        config={"configurable": {"thread_id": f"reporter-miss-{uuid.uuid4()}"}},
    )
    print("Research:
", miss_result["research"])
    print("
Writer output:
", miss_result["blog_post"])

---

## Closing

You built two LangGraph pipelines:
1. **Part A** — durable JSON notes + short-term checkpoints.
2. **Part B** — specialized nodes (Researcher → Writer) over a mock catalog.

**Next week:** Week 6 RAG uses vector search for document memory — a different tool for a different job.

## Challenges

### Challenge 01 — Add a catalog game
Add a new entry to `GAME_CATALOG` (name, genre, year, 3–5 facts) and run the reporter on it.

In [ ]:
# TODO
pass

### Challenge 02 — Editor node
Add an `editor` node after `writer` that tightens tone / shortens the draft. Recompile the graph and test on Celeste.

In [ ]:
# TODO
pass

### Challenge 03 — Favorite game in JSON
Use Part A's `save_note` to persist the student's favorite game, then ask the memory graph a follow-up that should mention it.

In [ ]:
# TODO
pass

### Challenge 04 (stretch) — Conditional edge
If research misses, route to a `clarify` node instead of `writer`. Hint: return different state and use `add_conditional_edges`.

In [ ]:
# TODO
pass